# Integrating Perple_X

**The workflow of modeling slab hydration will be as follows:**
1. We will use the tools built up in the SZ_problems notebooks to model and the pressure and velocity of the mantle wedge, the pressure and velocity of the slab, and temperature of the whole subduction zone
2. We will probe the temperature and pressure at various paths along the slab, and create curves in pressure-temperature space for those paths on the slab
3. We will use Perple_X to generate hydration data in at various pressures and temperatures given different initial mineral compositions and hydration states
4. We will compare our pressure-temperature curves of the slab profiles against the hydration data in PT space generated by Perple_X in order to predict where dehyrdation will occur along each slab, as well was predict which slabs will retain water throughout subduction

In this notebook, we are setting up a way to use Perple_X within our Jupyter Notebooks so that we can generate the hydration data that we are interested in.

First we import the modules we need

In [1]:
import sys, os
basedir = ''
if "__file__" in globals(): basedir = os.path.dirname(__file__)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as pl
import subprocess
import pathlib
output_folder = pathlib.Path(os.path.join(basedir, "output"))
output_folder.mkdir(exist_ok=True, parents=True)
data_folder = pathlib.Path(os.path.join(basedir, os.pardir, "data", "perple_x_data"))

Create array of lithology file basenames.

In [3]:
basenames = ["AlaskaTurb46_25", "Carbonate41_25", "Chert42_25", 
             "DMMdamp_25" , "DMMdry_25" , "DMMwet_25" ,
             "Diatom44_25" , "Pelagic45_25" , "Terrigenous43_25" ,
             "dike_25" , "gabbro_25" , "lovolc_25" , "upvolc_25"]

#### Run vertex (the first and longest Perple_X command that performs the free energy minimization) for all given lithologies

In [ ]:
def run_vertex(basenames):
    for basename in basenames:
        input = str(data_folder / basename)
        stdout = open(output_folder / str('vertex_'+ basename + '.log'), 'w')
        stderr = open(output_folder / str('vertex_'+ basename + '.err'), 'w')
        subprocess.run(["vertex"], input=input, text=True, stdout=stdout, stderr=stderr)
        stdout.close()
        stderr.close()
        print(basename, " vertex run complete")

In [ ]:
run_vertex(basenames)

#### Run werami to interpolate the free energy minimization.

We select the options:
* basename: project name uses the same input files from vertex
* 2: 2D grid
* 36: all phase &/or system properties (system properties here) - try a more compact output would work, e.g. 6 or 8 or 25
* 1: gives one System summary per node, 3 gives that plus all phases on lines
* n: y to include fluid in modal properties
* y: to change grid definition from values in .dat to something else
* 473 1673: T bounds (K)
* 1000 80000: P bounds (bar)
* 241 396: T, P nodes, designed for convenient/even 5C/0.02GPa grid
* 0: end


In [ ]:
def run_werami(basenames):
    for basename in basenames:
        input=str(data_folder / basename)+"""
        2
        36
        1
        n
        y
        473 1673
        1000 80000
        241 396
        0
        """
        stdout = open(output_folder / str('werami_'+ basename + '.log'), 'w')
        stderr = open(output_folder / str('werami_'+ basename + '.err'), 'w')
        subprocess.run(["werami"], input=input, text=True, stdout=stdout, stderr=stderr)
        stdout.close()
        stderr.close()
        print(basename, " werami run complete")

Now we call this function, again using our list of basenames as our input

In [ ]:
run_werami(basenames)

#### Data and plotting
Now we need to actually parse the data that we're interested in from the files that vertex and werami have created.


Specifically, we need to extract the pressure, temperature, and $H_2O$ wt% values from the .tab files created by running werami:

In [ ]:
def get_PT_data_from_tabs(basename):

    cols = ["T(K)", "P(bar)", "H2O,wt%"] #these are the column headers we're searching for

    datafile = data_folder / str(basename + '_1.tab')

    #find the row of the .tab file that contains the header
    header_idx = None
    with open(datafile, 'r') as f:
        i = 0
        for line in f: #find the headers we're searching for
            if all([c in line for c in cols]):
                header_idx = i
                break
            i += 1

    # some sanity checks
    if header_idx is None:
        raise RuntimeError("Could not find header row")

    if header_idx < 1:
        raise RuntimeError("Unexpected number of header rows")
    
    #Read P, T, and H2O data and make a data frame
    df = pd.read_csv(datafile, sep=r"\s+", skiprows=header_idx-1, header=1, usecols=cols)

    #Reshape and return the data
    P = np.unique(df['P(bar)'].to_numpy())/10000.0
    T = np.unique(df['T(K)'].to_numpy()) - 273.15
    H2O = df['H2O,wt%'].to_numpy().reshape(len(P),len(T))

    print(basename ," data read")

    return P, T, H2O

We can call this function for each supported basename, or we can use a for loop to call it for all of them, which we do here in order to verify that the data looks as expected for all the files:

In [ ]:
data = []
for basename in basenames:
    data.append(get_PT_data_from_tabs(basename))

Now, if we want, we can plot the data we've just extracted to make sure it looks like we would expect it to:

In [ ]:
def plot_PT_data(basenames, data, h20_max=5.5, save_figs=True):
    i=0
    for basename in basenames:
        fig, ax = pl.subplots(figsize=(7, 4.5))

        #Values for setting up the color bar
        vmin = 0.0
        vmax = h20_max
        dv = 0.25

        levels = np.arange(vmin, vmax+dv, dv)
        c = ax.contourf(data[i][1], data[i][0], data[i][2], levels=levels, cmap="jet_r")
        cbar = fig.colorbar(c, label=r"H$_2$O (wt%)")
        cbar.set_ticks(np.arange(vmin, vmax, 1, dtype=np.int32))
        ax.set_ylabel(r"P (GPa)")
        ax.set_xlabel(r"T ($^\circ$C)")
        ax.set_box_aspect(1)
        ax.set_title(basename)
        if save_figs == True:
            fig.savefig(output_folder / str(basename + '_PT_grid.png'))
        i += 1

Now we can call the function to visualize the data:

In [ ]:
plot_PT_data(basenames, data, save_figs=False)
#Set save_figs to True if you would like to automatically save generated figures to the output folder

: 

We now have a way to predict the water content of various lithologies at different pressures and temperatures. In the next notebook, we'll use the temperature probing functionality developed in the SZ_problems section to generate PT curves for sections of slabs and use those in conjuction with our PT hydration data.